# Experiment: Run Weather Baselines and GNN

Objective:
- train the comparison models on the same fixed month split
- produce artifacts that the comparison notebook can read

This notebook does not inspect model quality deeply. It just runs the jobs and records what finished.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path

import yaml

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
CONFIG_PATH = ROOT / 'configs' / 'default.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

print(ROOT)
print(CONFIG_PATH)


## Setup

The defaults below keep one fixed split and one output location per run type.
Change only the month lists or the boolean flags if you want a smoke test instead of a full run.

In [ ]:
TRAIN_MONTHS = None
VAL_MONTHS = None
SMOKE_MONTH = '199001'
RUN_SMOKE = True
RUN_INTERPOLATION = True
RUN_MLP = True
RUN_XGBOOST = True
RUN_GNN = True
DEVICE = CONFIG['training']['device']

if RUN_SMOKE:
    TRAIN_MONTHS = [SMOKE_MONTH]
    VAL_MONTHS = [SMOKE_MONTH]


In [ ]:
def build_month_args(train_months, val_months):
    args = []
    if train_months:
        args += ['--train-months', *train_months]
    if val_months:
        args += ['--val-months', *val_months]
    return args


def run_command(command):
    print(' '.join(command))
    completed = subprocess.run(command, cwd=ROOT, text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f'command failed: {completed.returncode}')
    return completed

month_args = build_month_args(TRAIN_MONTHS, VAL_MONTHS)
month_args


## Sanity Checks

This checks whether the processed dynamic tensors and final target tensors exist before running models.

In [ ]:
dynamic_dir = ROOT / CONFIG['era5']['processed_output_dir']
target_dir = ROOT / CONFIG['targets']['output_dir']
print('dynamic files', len(list(dynamic_dir.glob('era5_dynamic_*.pt'))))
print('target files ', len(list(target_dir.glob('targets_*.pt'))))
assert dynamic_dir.exists(), dynamic_dir
assert target_dir.exists(), target_dir


## Run Baselines

Interpolation uses the coarse inputs already stored in the processed tensors. MLP and XGBoost train from scratch on the same split.

In [ ]:
if RUN_INTERPOLATION:
    run_command([
        'python', 'src/training/train_baseline.py',
        '--model', 'interpolation',
        *month_args,
    ])

if RUN_MLP:
    run_command([
        'python', 'src/training/train_baseline.py',
        '--model', 'mlp',
        '--device', DEVICE,
        *month_args,
    ])

if RUN_XGBOOST:
    run_command([
        'python', 'src/training/train_baseline.py',
        '--model', 'xgboost',
        *month_args,
    ])


## Run GNN

This calls the existing training script. For a smoke test the config batch size and epoch count may still be too large, so override them here if needed.

In [ ]:
GNN_EPOCHS = 2 if RUN_SMOKE else CONFIG['training']['epochs']
GNN_BATCH_SIZE = 1

if RUN_GNN:
    run_command([
        'python', 'src/training/train.py',
        '--epochs', str(GNN_EPOCHS),
        '--batch-size', str(GNN_BATCH_SIZE),
        '--device', DEVICE,
        *month_args,
    ])


## Artifact Summary

This is the handoff to the comparison notebook.

In [ ]:
artifacts = {
    'baseline_interpolation': ROOT / CONFIG['baselines']['output_dir'] / 'interpolation' / 'metrics.json',
    'baseline_mlp': ROOT / CONFIG['baselines']['output_dir'] / 'mlp' / 'metrics.json',
    'baseline_xgboost': ROOT / CONFIG['baselines']['output_dir'] / 'xgboost' / 'metrics.json',
    'gnn_best': ROOT / CONFIG['training']['checkpoint_dir'] / 'best.pt',
    'gnn_last': ROOT / CONFIG['training']['checkpoint_dir'] / 'last.pt',
}
for name, path in artifacts.items():
    print(name, path.exists(), path)
